# Semantic Benchmark Java - Descriptive Spectral Analysis

This notebook is a descriptive visual validation report for one spectral clone-detection sub-dataset.

It intentionally focuses on interpretability rather than only pass/fail checks:

1. Three random source snippets are displayed beside their extracted Joern graph.
2. Clone and non-clone pairs are inspected with code, graph views, and eigenvalue arrays.
3. PSS scores and raw Wasserstein distances are plotted as clone/non-clone distributions.
4. A full threshold sweep searches for the best separation threshold and prints precision, recall, F1, accuracy, and ROC-AUC.


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "pipelines").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_ROOTS = {
    "project_root": str(PROJECT_ROOT),
    "data_root": os.environ.get("DATA_ROOT", os.path.join(os.path.dirname(str(PROJECT_ROOT)), "data")),
    "outputs_root": os.environ.get("OUTPUT_BASE_DIR", os.path.join(os.path.dirname(str(PROJECT_ROOT)), "outputs")),
}

from spectral_code.evaluation.notebook_helpers import (
    bcb_spec,
    xglue_spec,
    semantic_spec,
    artifact_status_dataframe,
    configure_notebook_style,
    notebook_output_dir,
    display_code_graph_side_by_side_examples,
    display_clone_nonclone_pair_inspection,
    display_statistical_distribution_plots,
    display_global_threshold_tuning_summary,
)

configure_notebook_style()
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:
spec = semantic_spec("Java")
GRAPH_TYPES = ['ast', 'cfg', 'ddg', 'pdg', 'cpg']
PRIMARY_GRAPH_TYPE = "cpg"
ANALYSIS_OUTPUT_DIR = notebook_output_dir(spec, CONFIG_ROOTS)

print("Data root:", CONFIG_ROOTS["data_root"])
print("Outputs root:", CONFIG_ROOTS["outputs_root"])
print("Notebook artifacts:", ANALYSIS_OUTPUT_DIR)
display(artifact_status_dataframe(spec))


## Visual Step 1 - Code vs. Graph Side-by-Side

This cell samples three random methods/snippets from the prepared split. For each example, the source code is rendered as monospaced text beside the selected Joern graph layer. Empty or missing graphs are logged instead of interrupting the notebook.


In [ ]:
random_examples = display_code_graph_side_by_side_examples(
    spec,
    n_examples=3,
    graph_type=PRIMARY_GRAPH_TYPE,
    seed=42,
    max_code_lines=38,
    max_nodes=80,
)


## Visual Step 2 - Clone vs. Non-Clone Pair Inspection

This section selects true clone pairs and true non-clone pairs with usable spectral features. For each pair, it displays code snippet 1 vs. code snippet 2, their graphs, and a line plot of the eigenvalue arrays so structural similarity or separation can be inspected visually.


In [ ]:
inspected_pairs = display_clone_nonclone_pair_inspection(
    spec,
    graph_type=PRIMARY_GRAPH_TYPE,
    per_label=2,
    seed=42,
    max_code_lines=70,
    max_nodes=80,
)


## Visual Step 3 - Statistical Distribution Plots

This cell computes PSS similarity and raw Wasserstein distance for the sub-dataset. Clone distributions are shown in green and non-clone distributions in red, with density overlays to make the separability margin visible.


In [ ]:
score_df = display_statistical_distribution_plots(
    spec,
    sample_size=None,
    graph_types=GRAPH_TYPES,
    seed=42,
)


## Visual Step 4 - Global Hyperparameter Tuning & Metrics Summary

The final cell sweeps candidate thresholds across the full score table, chooses the best threshold by F1, and prints a clean classification report plus accuracy and ROC-AUC.


In [ ]:
threshold_sweep_df, best_threshold_config = display_global_threshold_tuning_summary(
    spec,
    graph_types=GRAPH_TYPES,
    optimize_for="f1",
    seed=42,
)
